# Single-field ALMA simulation

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/simulation/alma_single_field_simulation.ipynb)

Port of the SIRIUS `alma_single_field` notebook: an ALMA 12 m array observation of one and then
four point sources placed by **image pixel**, simulated with an Airy-disk beam and imaged (dirty
and CLEANed) with AstroVIPER.

---
## API


In [ ]:
from astroviper.distributed_applications.simulation import simulate_processing_set

simulate_processing_set?

## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from astropy.coordinates import SkyCoord

xr.set_options(display_style="html")
ARCSEC_TO_RAD = np.pi / (180 * 3600)

In [ ]:
from toolviper.dask.client import local_client

viper_client = local_client(cores=4, memory_limit="4GB")
viper_client

## ALMA 12 m array

Select the 12 m dishes of the full ALMA layout.

In [ ]:
from astroviper.utils.telescope_layout import read_telescope_layout

alma_all = read_telescope_layout("alma.all")
is_12m = alma_all.ANTENNA_DISH_DIAMETER.values == 12.0
antenna_xds = alma_all.isel(antenna_name=np.where(is_12m)[0][:40])
n_antenna = antenna_xds.sizes["antenna_name"]
print(n_antenna, "antennas")

In [ ]:
time_params = {
    "time_start": "2019-10-03T19:00:00.000",
    "time_delta": 2000.0,
    "n_samples": 18,
}
frequency_params = {
    "freq_start": 90e9,
    "freq_delta": 0.5e9,
    "n_channels": 3,
    "channel_width": 0.5e9,
    "spectral_window_name": "Band3",
}
polarization = ["XX", "YY"]

from astroviper.utils.beam_models import airy_disk_model

beam_models = [airy_disk_model("alma", func="airy")]
beam_model_map = np.zeros(n_antenna, dtype=int)

## Place sources by pixel

``sin_pixel_to_celestial_coord`` converts pixel positions of a SIN-projected image (size, cell)
centred on the phase centre to sky positions.

In [ ]:
from astroviper.utils.coordinate_transforms import sin_pixel_to_celestial_coord

image_size = np.array([512, 512])
cell_size_arcsec = 0.3
cell_size = np.array([-cell_size_arcsec, cell_size_arcsec]) * ARCSEC_TO_RAD

phase_center = SkyCoord(ra="19h59m28.5s", dec="-40d44m01.5s", frame="icrs")
phase_center_ra_dec = np.array([phase_center.ra.rad, phase_center.dec.rad])[None, :]

pixels_single = np.array([[256, 256]])
point_source_ra_dec = sin_pixel_to_celestial_coord(
    phase_center_ra_dec[0], image_size, cell_size, pixels_single
)[None, :, :]
point_source_flux = np.array([1.0, 0, 0, 1.0])[None, None, None, :]

In [ ]:
result = simulate_processing_set(
    ps_store="alma_sim.ps.zarr",
    antenna_xds=antenna_xds,
    time_params=time_params,
    frequency_params=frequency_params,
    polarization=polarization,
    point_source_flux=point_source_flux,
    point_source_ra_dec=point_source_ra_dec,
    phase_center_ra_dec=phase_center_ra_dec,
    beam_models=beam_models,
    beam_model_map=beam_model_map,
    n_time_chunks=3,
    n_frequency_chunks=3,
    overwrite=True,
)
result["timing_node_tasks"][["task_id", "T_uvw", "T_visibilities", "T_write"]]

## Validate the processing set against the MSv4 schema

``xradio.schema.check.check_datatree`` checks every dataset of the processing set (coordinates,
dimensions, dtypes and attributes of the main, antenna and field/source datasets) against the
MSv4 schema.  ``simulate_processing_set`` runs this check itself (``check_schema=True``) and logs
a warning on problems; here it is run explicitly so that the result is visible.

In [ ]:
from xradio.measurement_set import open_processing_set
from xradio.schema.check import check_datatree

ps_xdt = open_processing_set("alma_sim.ps.zarr")
issues = check_datatree(ps_xdt)
print(issues)
assert str(issues) == "No schema issues found"
# the same check works on a single MSv4 (the checker dispatches on the ``type`` attribute)
print(check_datatree(ps_xdt[result["ms_name"]]))

## Dirty and cleaned images

In [ ]:
from xradio.image import load_image
from xradio.measurement_set import open_processing_set

from astroviper.distributed_applications.imaging import image_cube_single_field


def image_simulation(
    ps_store,
    image_store,
    image_size,
    cell_size_arcsec,
    niter=0,
    polarization_coords=("I",),
    n_chunks=2,
):
    """Make a (dirty or cleaned) cube of a simulated processing set with AstroVIPER."""
    ps_xdt = open_processing_set(ps_store)
    combined = ps_xdt.xr_ps.get_combined_field_and_source_xds()
    phase_direction = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
        field_name=combined.attrs["center_field_name"]
    ).values
    image_params = {
        "image_size": list(image_size),
        "cell_size": np.array([-cell_size_arcsec, cell_size_arcsec]) * ARCSEC_TO_RAD,
        "phase_direction": phase_direction,
        "frequency_coords": ps_xdt.xr_ps.get_freq_axis().values,
        "polarization_coords": list(polarization_coords),
        "time_coords": [0],
        "fft_padding": 1.2,
        "cpp_gridder": True,
    }
    iteration_control = {
        "niter": niter,
        "nmajor": -1 if niter > 0 else 0,
        "threshold": 0.0,
        "gain": 0.1,
        "cyclefactor": 1.5,
        "cycleniter": -1,
        "minpsffraction": 0.05,
        "maxpsffraction": 0.8,
        "primary_beam_limit": 0.1,
    }
    keep = [
        "sky_residual",
        "point_spread_function",
        "primary_beam",
        "beam_fit_params_point_spread_function",
    ]
    if niter > 0:
        keep += ["sky_model", "mask"]
    image_cube_single_field(
        ps_store=ps_store,
        image_store=image_store,
        image_params=image_params,
        imaging_weights_params={
            "weighting": "natural",
            "robust": 0.5,
            "casa_weighting_implementation": True,
        },
        iteration_control_params=iteration_control,
        gridder="prolate_spheroidal",
        deconvolver="hogbom_many_threads",
        scan_intents="OBSERVE_TARGET#ON_SOURCE",
        image_data_variables_keep=keep,
        processing_set_data_group_name="base",
        single_precision_image=False,
        processing_function_threads=1,
        n_chunks=n_chunks,
        overwrite=True,
        restore=niter > 0,
    )
    return load_image(image_store)


def show_image(
    img_xds, variable="SKY_RESIDUAL", frequency=0, polarization=0, title=None, vmax=None
):
    """Plot one plane of an AstroVIPER image with l/m in arcsec."""
    plane = (
        img_xds[variable]
        .isel(time=0, frequency=frequency, polarization=polarization)
        .values
    )
    extent = (
        np.array(
            [
                img_xds.l.values[0],
                img_xds.l.values[-1],
                img_xds.m.values[0],
                img_xds.m.values[-1],
            ]
        )
        / ARCSEC_TO_RAD
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(plane.T, origin="lower", extent=extent, cmap="viridis", vmax=vmax)
    ax.set_xlabel("l [arcsec]")
    ax.set_ylabel("m [arcsec]")
    ax.set_title(
        title
        or f"{variable} channel {frequency} ({img_xds.frequency.values[frequency] / 1e9:.3f} GHz)"
    )
    fig.colorbar(im, ax=ax, label="Jy/beam")
    return fig

In [ ]:
dirty = image_simulation(
    "alma_sim.ps.zarr", "alma_sim_dirty.img.zarr", image_size, cell_size_arcsec, niter=0
)
show_image(dirty, title="dirty image, single source at the phase centre")
plt.show()
clean = image_simulation(
    "alma_sim.ps.zarr",
    "alma_sim_clean.img.zarr",
    image_size,
    cell_size_arcsec,
    niter=1000,
)
show_image(clean, variable="SKY_RESTORED", title="restored image (niter=1000)")
plt.show()
print("restored peak [Jy/beam]:", clean.SKY_RESTORED.values.max())

## Four sources

Sources at different offsets experience different primary-beam attenuation.

In [ ]:
pixels_multi = np.array([[256, 256], [256, 356], [156, 156], [356, 256]])
point_source_ra_dec_multi = sin_pixel_to_celestial_coord(
    phase_center_ra_dec[0], image_size, cell_size, pixels_multi
)[None, :, :]
point_source_flux_multi = np.tile(np.array([1.0, 0, 0, 1.0]), (4, 1, 1, 1))

result = simulate_processing_set(
    ps_store="alma_msource_sim.ps.zarr",
    antenna_xds=antenna_xds,
    time_params=time_params,
    frequency_params=frequency_params,
    polarization=polarization,
    point_source_flux=point_source_flux_multi,
    point_source_ra_dec=point_source_ra_dec_multi,
    phase_center_ra_dec=phase_center_ra_dec,
    beam_models=beam_models,
    beam_model_map=beam_model_map,
    n_time_chunks=3,
    n_frequency_chunks=3,
    overwrite=True,
)
clean_multi = image_simulation(
    "alma_msource_sim.ps.zarr",
    "alma_msource_clean.img.zarr",
    image_size,
    cell_size_arcsec,
    niter=2000,
)
show_image(clean_multi, variable="SKY_RESTORED", title="restored image, four sources")
plt.show()
restored = clean_multi.SKY_RESTORED.isel(time=0, frequency=0, polarization=0).values
pb = clean_multi.PRIMARY_BEAM.isel(time=0, frequency=0, polarization=0).values
for pix in pixels_multi:
    print(
        f"pixel {pix}: restored {restored[pix[0], pix[1]]:.3f} Jy/beam, primary beam {pb[pix[0], pix[1]]:.3f}"
    )

## Clean up

In [ ]:
import shutil

for path in [
    "alma_sim.ps.zarr",
    "alma_msource_sim.ps.zarr",
    "alma_sim_dirty.img.zarr",
    "alma_sim_clean.img.zarr",
    "alma_msource_clean.img.zarr",
]:
    shutil.rmtree(path, ignore_errors=True)
viper_client.close()